<!-- beginner-banner-v1 -->

> 🧭 **비개발자 수강생 안내** — 이 노트북에서 새로 배우는 것: 본인 에이전트 실행을 **재생 영상처럼 추적** — LangSmith 트레이싱.
>
> - 📖 강의 페이지: [day4/21-langsmith](https://siapapa.github.io/day4/21-langsmith/)
> - 🆕 처음이라면 → [비개발자 학습 가이드](https://siapapa.github.io/beginners-guide/)
> - 🔤 모르는 단어 → [용어 사전](https://siapapa.github.io/appendix/glossary/)
> - 🛠️ 환경/접속 막힘 → [사전 준비](https://siapapa.github.io/setup/) · [트러블슈팅](https://siapapa.github.io/appendix/troubleshooting/)
>
> **셀은 위에서 아래로 차례대로 실행**하세요. 시연용 코드(`구경만 하세요` 표시)는 지금 이해 못 해도 100% 정상입니다.

---



# 18. LangSmith 트레이싱 — 에이전트 관측 가능성
> Day 4 · 21H · 소요 약 50분

## 학습 목표

- LangSmith의 역할(LLM 애플리케이션 관측 가능성)을 설명할 수 있다.
- Run/Trace 구조를 이해하고 에이전트 실행을 자동으로 트레이싱할 수 있다.
- 토큰 사용량 / 지연 / 비용을 LangSmith API로 분석한다.
- 평가용 Dataset을 프로그래밍 방식으로 생성하고 Feedback을 부여한다.

> **전제 노트북:** 17번(`17_my_sql_agent.ipynb`)의 SQL 에이전트를 이 노트북에서 다시 이식합니다. 로직은 완전히 동일하며, LangSmith 트레이싱을 활성화해 각 노드·재시도·토큰·비용을 가시화합니다.
> **필요 키:** `OPENAI_API_KEY`, `NEON_DSN`, 그리고 **`LANGSMITH_API_KEY`** (LangSmith 무료 계정, https://smith.langchain.com/).

In [ ]:
%pip install -q langgraph langchain langchain-openai langsmith sqlalchemy psycopg2-binary sqlparse pandas matplotlib tabulate

In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다 (다른 노트북과 동일 패턴).
import os


def _load_secret(key: str, required: bool = True) -> None:
    """Colab Secrets → getpass 입력 순으로 시도해 환경변수에 적재."""
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")


# 이 노트북은 OpenAI(LLM) + Neon(DB) + LangSmith(트레이싱) 세 키가 모두 필요합니다.
_load_secret("OPENAI_API_KEY", required=True)
_load_secret("NEON_DSN", required=True)
_load_secret("LANGSMITH_API_KEY", required=True)

# ── LangSmith 자동 트레이싱 활성화 ─────────────────────────────────────
# 핵심 아이디어: 코드에 "트레이싱 코드" 를 넣지 않습니다. 대신 환경변수 몇 개만 설정해 두면
# LangChain/LangGraph 가 모든 호출을 자동으로 LangSmith 서버에 전송해 줍니다.
os.environ["LANGCHAIN_TRACING_V2"] = "true"        # 트레이싱 ON
os.environ["LANGCHAIN_PROJECT"] = os.environ.get("LANGCHAIN_PROJECT", "sql-agent-day4")  # 그룹 이름

# langsmith SDK 0.1.70 부터 환경변수 이름이 LANGSMITH_* 로 바뀌고 있습니다.
# 두 이름을 모두 세팅해 둬야 신구 버전 모두 호환됩니다 (어느 한쪽만 읽는 코드가 있어도 동작).
os.environ["LANGCHAIN_API_KEY"] = os.environ["LANGSMITH_API_KEY"]
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = os.environ["LANGCHAIN_PROJECT"]

print("Environment ready.")
print(f"  LANGCHAIN_TRACING_V2 = {os.environ.get('LANGCHAIN_TRACING_V2')}")
print(f"  LANGCHAIN_PROJECT    = {os.environ.get('LANGCHAIN_PROJECT')}")
print(f"  LANGSMITH_API_KEY    = {'설정됨' if os.environ.get('LANGSMITH_API_KEY') else '미설정'}")

## 1. LangSmith란 무엇인가

전통 소프트웨어와 달리 LLM 애플리케이션은 **비결정적**입니다. 같은 질문에도 SQL 이 매번 다르게 생성될 수 있고, 실패 원인도 프롬프트 · 컨텍스트 · 모델 파라미터가 얽혀서 발생합니다. **관측 가능성(Observability)** 없이는 디버깅이 사실상 불가능합니다.

| 구분 | 전통 소프트웨어 | LLM 애플리케이션 |
|---|---|---|
| 출력 | 결정적 | 비결정적 |
| 버그 재현 | 쉬움 | 어려움 |
| 디버깅 도구 | 로그 / 디버거 | **트레이스 + 토큰 + 프롬프트 전문** |
| 평가 방법 | 단위 테스트 | 정량 평가 프레임워크 (Ragas 등) |

### Run / Trace 구조

```
Trace (에이전트 실행 1건)
├─ Run: generate_sql     (LLM 호출)
│    └─ input / output / tokens / latency / cost
├─ Run: execute_sql      (도구 실행)
├─ Run: validate_sql     (함수 실행)
└─ Run: generate_answer  (LLM 호출)
```

LangSmith 웹 UI (https://smith.langchain.com/) 에서는 **Trace 목록 → 개별 Trace 상세 → Run Tree → Analytics(통계)** 4단 메뉴로 이 데이터를 탐색합니다. 우리는 UI에서 눈으로 확인하는 것과 동시에, 이 노트북에서는 **`langsmith.Client` API**로 프로그램적으로 뽑아 분석합니다.

In [ ]:
# ============================================================
# 1. LangSmith Client 생성 + 프로젝트 확인
# ============================================================
# TODO: `from langsmith import Client`로 클라이언트를 생성하고, 현재 `LANGSMITH_PROJECT` 값을 출력해 확인하세요.
# 여기에 구현하세요.


## 2. 17번 에이전트를 재이식

아래 셀들에서 17번 노트북의 에이전트를 **self-contained** 로 다시 세웁니다. 로직(상태·가드레일·4노드·재시도 분기)은 17번과 1:1 동일합니다. 차이는 환경변수로 **LangSmith 자동 트레이싱**이 켜져 있다는 것 뿐 — 별도 계측 코드는 필요 없습니다.

In [ ]:
# ============================================================
# 2. 17번 에이전트 재이식 — 의존성 import + DB 엔진
# ============================================================
# TODO: `sqlalchemy.create_engine`으로 Neon 엔진을 만들고(읽기 전용 옵션 시도 후 실패 시 일반 모드로 폴백), `langgraph`/`langchain_openai`/`langchain_core` 모듈을 import 하세요.
# 여기에 구현하세요.


In [ ]:
# ============================================================
# 스키마 수집 함수
# ============================================================
# TODO: `sqlalchemy.inspect`로 컬럼·FK를 가져와 `CREATE TABLE` 문자열을 만들고, `information_schema.columns` + `col_description`으로 COMMENT를 붙이는 `collect_schema(engine, tables)`를 작성하세요.
# 여기에 구현하세요.


In [ ]:
# ============================================================
# AgentState + 가드레일
# ============================================================
# TODO: `TypedDict`로 `question/sql/sql_result/error/answer/attempts` 필드를 가진 `AgentState`를 정의하고, 정규식으로 DDL/DML(`DROP|DELETE|UPDATE|INSERT|ALTER|TRUNCATE|CREATE`) 키워드를 차단하는 `sanitize_sql`을 만드세요.
# 여기에 구현하세요.


In [ ]:
# ============================================================
# 노드 함수들
# ============================================================
# TODO: Day 3과 동일하게 `generate_sql` (프롬프트→LLM→파싱), `run_sql` (`sanitize_sql`+`pd.read_sql`), `validate`, `answer` (결과 요약 프롬프트), `should_retry` (조건부 분기) 5개 노드를 작성하세요.
# 여기에 구현하세요.


In [ ]:
# ============================================================
# 그래프 조립
# ============================================================
# TODO: `StateGraph(AgentState)`로 4개 노드를 추가하고 `generate_sql → run_sql → validate`로 엣지를 연결, `validate`에 `should_retry` 조건부 엣지를 걸어 `compile()` 하세요.
# 여기에 구현하세요.


## 3. 10개 질문 일괄 실행 — 자동 트레이싱

`LANGCHAIN_TRACING_V2="true"` 가 켜져 있으면, 모든 LangChain / LangGraph 실행이 **자동으로** LangSmith에 기록됩니다. 코드에 아무 계측도 추가할 필요가 없습니다.

In [ ]:
# ============================================================
# 3. 10개 질문 일괄 실행 — 자동 트레이싱
# ============================================================
# TODO: 10개 질문 리스트를 순회하며 `agent.invoke({"question": q, "attempts": 0})`를 호출하고, question/status/attempts/sql/answer/sql_result를 `results` 리스트에 모은 뒤 정답률을 출력하세요.
# 여기에 구현하세요.


## 4. LangSmith API로 트레이스 분석

LangSmith UI에서 눈으로 보는 것 뿐만 아니라, `list_runs` 로 프로그램적으로 모든 실행 데이터를 끌어올 수 있습니다. 여기서는 질문별 토큰 / 지연 / 예상 비용을 DataFrame으로 만들어 시각화합니다.

> **주의:** LangSmith 서버에 실행 데이터가 도착하기까지 보통 수 초가 걸립니다. 아래 셀에서 `list_runs` 결과가 비어 있으면 15~30초 대기 후 재실행하세요.

In [ ]:
# ============================================================
# 4. LangSmith API로 트레이스 분석 — root run 수집
# ============================================================
# TODO: `Client().list_runs(project_name=..., is_root=True, limit=10)`로 root run을 가져오세요(구버전 SDK는 `execution_order=1`로 폴백).
# 여기에 구현하세요.


In [ ]:
# ============================================================
# 토큰/비용/지연 DataFrame 구성
# ============================================================
# TODO: 각 run의 `total_tokens`/`extra["runtime"]["token_usage"]`에서 토큰을 방어적으로 추출하여 토큰·지연(`end_time-start_time`)·비용을 DataFrame으로 정리하세요.
# 여기에 구현하세요.


## 5. 시각화 — 질문별 토큰/지연 2-패널 차트

막대 길이로 어떤 질문이 토큰을 많이 쓰고 어떤 질문이 느린지를 한눈에 파악합니다. 재시도가 발생한 질문은 보통 토큰·지연이 모두 큰 봉우리로 나타납니다.

In [ ]:
# ============================================================
# 5. 시각화 — 질문별 토큰/지연 2-패널 차트
# ============================================================
# TODO: `matplotlib`의 `subplots(1, 2)`와 `barh`로 좌측에 토큰, 우측에 지연 시간을 가로 막대 차트로 그리고 `savefig`로 PNG 저장하세요.
# 여기에 구현하세요.


### LangSmith UI 연계 실습

1. 위 차트에서 **지연이 가장 긴 질문**을 하나 고르세요.
2. https://smith.langchain.com/ 에 로그인 → 현재 프로젝트(`sql-agent-day4`) 를 엽니다.
3. 해당 질문의 Trace를 클릭 → **Run Tree** 화면을 엽니다.
4. `generate_sql` / `execute_sql` / `validate_sql` / `generate_answer` 노드의 소요 시간을 비교하고 병목 노드를 찾으세요.
5. 재시도가 있었다면 `generate_sql` 가 2번 이상 나타날 겁니다 — 첫 번째와 두 번째의 프롬프트 / 출력 차이를 비교하세요.

> **FAQ — "UI에 아무것도 안 보여요"**
>
> - `LANGSMITH_API_KEY` 를 다시 확인하세요 (만료 / 오타 가능).
> - 처음 트레이스가 서버에 반영되기까지 최대 1분 지연될 수 있습니다.
> - Colab 무료 계정은 네트워크가 가끔 끊깁니다 — `runs` 가 0개면 15~30초 뒤 위 셀만 재실행.

## 6. 평가용 Dataset 생성

LangSmith **Dataset** = "질문 + 기대 정답(ground truth)" 의 모음. 이 Dataset은 다음 노트북(`19_ragas_eval.ipynb`) 에서 Ragas 정량 평가의 ground_truth로 재사용됩니다.

아래 셀은:
1. 같은 이름의 데이터셋이 이미 있으면 **삭제 후 재생성** (멱등 실행).
2. 질문 10개 + 병원 DB 기준 간단 ground_truth 를 `create_example` 로 등록.

In [ ]:
# ============================================================
# 6. 평가용 Dataset 생성
# ============================================================
# TODO: `ls.create_dataset(dataset_name=...)`으로 데이터셋을 만들고, 질문/정답 쌍 10개를 `ls.create_example(inputs=..., outputs=..., dataset_id=...)`로 등록하세요. 동명 데이터셋이 있으면 `read_dataset` + `delete_dataset`으로 먼저 정리하면 됩니다.
# 여기에 구현하세요.


## 7. Feedback 태깅 — 성공/실패 라벨 부여

**Feedback** = 개별 Run에 점수·코멘트를 붙이는 기능. 정답/오답을 태깅해 두면 LangSmith UI에서 "실패한 Run 만" 필터링해 디버깅할 수 있습니다.

> **주의:** `list_runs()` 는 기본적으로 **최신 → 과거** 역순입니다. 그냥 `zip(runs, results)` 하면 엉뚱한 Run에 Feedback이 붙습니다. 아래는 **질문 문자열로 매칭** 하는 안전한 방법.

In [ ]:
# ============================================================
# 7. Feedback 태깅 — 성공/실패 라벨 부여
# ============================================================
# TODO: `list_runs(is_root=True)`는 최신→과거 역순이므로 `start_time`으로 정렬하거나 질문 문자열로 매칭한 뒤, 각 run에 `ls.create_feedback(run_id=..., key="correctness", score=1.0 또는 0.0, comment=...)`로 정답/오답 태그를 부여하세요.
# 여기에 구현하세요.


## 실습 과제

1. **본인 에이전트에 LangSmith 트레이싱을 연결**하고 10개 질문을 실행하세요.
2. LangSmith UI에서 **가장 느린 질문의 트레이스를 열어 병목 노드를 찾으세요**.
    - 어떤 노드가 가장 오래 걸렸나요?
    - 재시도(retry)가 발생했나요?
3. **평가용 Dataset을 본인 질문 10개로 생성**하세요.
4. (도전) Feedback으로 정답/오답을 태깅하고, LangSmith UI에서 실패 케이스만 필터링해보세요.

**생각해보기**

- LangSmith 트레이싱 없이 에이전트를 디버깅하려면 어떻게 해야 할까요? print 문을 곳곳에 넣는 방법과 비교하면 어떤 장점이 있나요?
- 토큰 사용량이 가장 많은 질문과 가장 적은 질문의 차이는 무엇인가요? 어떤 유형의 질문이 토큰을 많이 소비하나요?
- 프로덕션 환경에서 LangSmith를 사용할 때, 민감한 데이터(환자 정보 등)를 어떻게 처리해야 할까요?


In [ ]:
# ============================================================
# 실습 과제 — 본인 프로젝트에 LangSmith 연결
# ============================================================
# TODO: `LANGCHAIN_PROJECT`(또는 `LANGSMITH_PROJECT`)를 본인 이름(예: `sql-agent-{yourname}`)으로 바꾸고, 본인 도메인 질문 10개로 `agent.invoke`를 호출한 뒤 LangSmith UI에서 trace URL을 복사해 과제 #3 제출에 포함하세요.
# 여기에 구현하세요.


## 다음 노트북에서는...

`19_ragas_eval.ipynb` 에서는 이 에이전트가 "정확한 답을 내는지" 를 **정량 평가** 합니다. LangSmith가 "어떻게 실행되었는가"(지연·토큰·분기)를 보여줬다면, Ragas는 "얼마나 좋은 답이었는가"(Faithfulness / Answer Relevancy / Context Precision·Recall)를 4개 지표로 채점합니다. 프롬프트 튜닝 전/후 비교 차트를 만들어 최종 발표의 핵심 슬라이드로 활용합니다.